# Publish Model Showcase

This notebook does **not** train the model. It reads the completed persistent artifacts from Google Drive, generates committee-ready evaluation charts and a measured model card, then uploads only the showcase files to the existing Hugging Face model repository.


In [ ]:
# 1. Load the current project source — CPU is sufficient
from pathlib import Path
from urllib.request import Request, urlopen
import io, json, os, shutil, subprocess, sys, zipfile

SHOWCASE_RELEASE = "2026.09.28-s1"
REPOSITORY_ID = 1391259908
headers = {"User-Agent": "semantic-integrity-showcase", "Accept": "application/vnd.github+json"}

commit_request = Request(
    f"https://api.github.com/repositories/{REPOSITORY_ID}/commits/main",
    headers=headers,
)
with urlopen(commit_request, timeout=30) as response:
    commit_sha = json.load(response)["sha"]

archive_request = Request(
    f"https://api.github.com/repositories/{REPOSITORY_ID}/zipball/{commit_sha}",
    headers=headers,
)
with urlopen(archive_request, timeout=120) as response:
    archive_bytes = response.read()

workspace = Path("/content/semantic_integrity_showcase")
extract_root = Path("/content/semantic_integrity_showcase_source")
for path in (workspace, extract_root):
    if path.exists():
        shutil.rmtree(path)
extract_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    archive.extractall(extract_root)
roots = [path for path in extract_root.iterdir() if path.is_dir()]
if len(roots) != 1:
    raise RuntimeError("Unexpected repository archive layout")
shutil.move(str(roots[0]), str(workspace))
os.chdir(workspace)
print("Showcase release:", SHOWCASE_RELEASE)
print("Source commit:", commit_sha[:12])


In [ ]:
# 2. Mount persistent model artifacts and install only showcase dependencies
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import importlib.util
missing = []
if importlib.util.find_spec("matplotlib") is None:
    missing.append("matplotlib>=3.8")
if importlib.util.find_spec("huggingface_hub") is None:
    missing.append("huggingface_hub>=0.35,<1")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", *missing], check=True)

cache_root = Path("/content/drive/MyDrive/semantic_integrity_model_cache")
assert cache_root.exists(), f"Persistent model cache not found: {cache_root}"
print("Persistent cache: PASS")


In [ ]:
# 3. Generate measured charts and the committee-ready model card
showcase_dir = Path("artifacts/hf_showcase")
result = subprocess.run(
    [sys.executable, "-m", "scripts.build_hf_showcase", "--cache-root", str(cache_root), "--output", str(showcase_dir)],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Showcase generation failed")

required = [
    "README.md",
    "evaluation_metrics.json",
    "dataset_summary.json",
    "dataset_qa_summary.json",
    "showcase/metrics_overview.png",
    "showcase/false_safe_rate.png",
    "showcase/per_label_f1.png",
    "showcase/label_distribution.png",
    "showcase/split_distribution.png",
]
missing_files = [name for name in required if not (showcase_dir / name).exists()]
assert not missing_files, f"Missing showcase files: {missing_files}"
print("Showcase generation: PASS")
print("Files:")
for name in required:
    print(" -", name)


In [ ]:
# 4. Upload the showcase to the existing Hugging Face model repository
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets")

api = HfApi(token=hf_token)
identity = api.whoami()
namespace = identity.get("name")
if not namespace:
    raise RuntimeError("Could not resolve the Hugging Face account namespace")
repo_id = f"{namespace}/semantic-integrity-v0-1"
print("Authenticated account:", namespace)
print("Target repository:", repo_id)

api.repo_info(repo_id=repo_id, repo_type="model", token=hf_token)
api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path=str(showcase_dir),
    token=hf_token,
    commit_message="Add measured evaluation showcase and committee visuals",
)
print("Hugging Face showcase upload: PASS")
print("Model page:", f"https://huggingface.co/{repo_id}")


In [ ]:
# 5. Verify that the showcase is visible in the model repository
info = api.model_info(repo_id=repo_id, token=hf_token)
siblings = {item.rfilename for item in info.siblings}
expected = {
    "README.md",
    "evaluation_metrics.json",
    "dataset_summary.json",
    "dataset_qa_summary.json",
    "showcase/metrics_overview.png",
    "showcase/false_safe_rate.png",
    "showcase/per_label_f1.png",
    "showcase/label_distribution.png",
    "showcase/split_distribution.png",
}
missing_remote = sorted(expected - siblings)
assert not missing_remote, f"Remote showcase verification failed: {missing_remote}"
print("REMOTE SHOWCASE VERIFICATION: PASS")
print("Everything needed for the committee model page is published.")
